In [1]:
print("Here begins the hybrid models notebook")

Here begins the hybrid models notebook


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
df_ml = pd.read_csv("../../data/processed/mental_health_tech_ml.csv")

df_ml.head()

,Age,self_employed,family_history,treatment,remote_work,tech_company,obs_consequence,Age__missing,Gender__missing,state__missing,self_employed__missing,work_interfere__missing,work_interfere_Never,work_interfere_Often,work_interfere_Rarely,work_interfere_Sometimes,Gender_male,Country_Austria,"Country_Bahamas, The",Country_Belgium,Country_Bosnia and Herzegovina,Country_Brazil,Country_Bulgaria,Country_Canada,Country_China,Country_Colombia,Country_Costa Rica,Country_Croatia,Country_Czech Republic,Country_Denmark,Country_Finland,Country_France,Country_Georgia,Country_Germany,Country_Greece,Country_Hungary,Country_India,Country_Ireland,Country_Israel,Country_Italy,Country_Japan,Country_Latvia,Country_Mexico,Country_Moldova,Country_Netherlands,Country_New Zealand,Country_Nigeria,Country_Norway,Country_Philippines,Country_Poland,Country_Portugal,Country_Romania,Country_Russia,Country_Singapore,Country_Slovenia,Country_South Africa,Country_Spain,Country_Sweden,Country_Switzerland,Country_Thailand,Country_United Kingdom,Country_United States,Country_Uruguay,Country_Zimbabwe,state_AZ,state_CA,state_CO,state_CT,state_DC,state_FL,state_GA,state_IA,state_ID,state_IL,state_IN,state_KS,state_KY,state_LA,state_MA,state_MD,state_ME,state_MI,state_MN,state_MO,state_MS,state_NC,state_NE,state_NH,state_NJ,state_NM,state_NV,state_NY,state_OH,state_OK,state_OR,state_PA,state_RI,state_SC,state_SD,state_TN,state_TX,state_UT,state_VA,state_VT,state_WA,state_WI,state_WV,state_WY,no_employees_100-500,no_employees_26-100,no_employees_500-1000,no_employees_6-25,no_employees_More than 1000,benefits_No,benefits_Yes,care_options_Not sure,care_options_Yes,wellness_program_No,wellness_program_Yes,seek_help_No,seek_help_Yes,anonymity_No,anonymity_Yes,leave_Somewhat difficult,leave_Somewhat easy,leave_Very difficult,leave_Very easy,mental_health_consequence_No,mental_health_consequence_Yes,phys_health_consequence_No,phys_health_consequence_Yes,coworkers_Some of them,coworkers_Yes,supervisor_Some of them,supervisor_Yes,mental_health_interview_No,mental_health_interview_Yes,phys_health_interview_No,phys_health_interview_Yes,mental_vs_physical_No,mental_vs_physical_Yes
0,37.0,0.0,0,1,0,1,0,0,0,0,1,0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,True,False,True,False,False,True,False,True,False,True,False,False,True,False,True,False,True,False,False,True,True,False,False,False,False,True
1,44.0,0.0,0,0,0,0,0,0,0,0,1,0,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,True,False,False,False
2,32.0,0.0,0,0,0,1,0,0,0,1,1,0,False,False,True,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,F

In [2]:
# Convert bool to integers
bool_cols = df_ml.select_dtypes(include=["bool"]).columns
df_ml[bool_cols] = df_ml[bool_cols].astype(int)

df_ml.dtypes.value_counts()

for col in df_ml.columns:
    if df_ml[col].dtype == "object":
        unique_vals = set(df_ml[col].dropna().astype(str).unique())
        if unique_vals.issubset({"True", "False"}):
            df_ml[col] = df_ml[col].map({"True": 1, "False": 0})

In [3]:
# Target variable: work interference + treatment
# Create a binary target variable for work interference
# 0 --> no interference, 1 --> interference (rarely/sometimes/often)

# df = df_ml.copy()
#
# df["work_interfere_binary"] = (
#     df[[
#         "work_interfere_Often",
#         "work_interfere_Rarely",
#         "work_interfere_Sometimes"
#     ]].sum(axis=1) > 0
# ).astype(int)
#
# # Create a combined target variable with three classes:
# # 0 --> no interference, 1 --> interference but no treatment, 2 --> interference
# df["combined_target"] = None
#
# df.loc[df["work_interfere_binary"] == 0, "combined_target"] = 0
#
# df.loc[
#     (df["work_interfere_binary"] == 1) &
#     (df["treatment"] == 0),
#     "combined_target"
# ] = 1
#
# df.loc[
#     (df["work_interfere_binary"] == 1) &
#     (df["treatment"] == 1),
#     "combined_target"
# ] = 2
#
# df["combined_target"] = df["combined_target"].astype(int)
#
# # df["combined_target"].value_counts()
# df["combined_target"].value_counts(normalize=True)

# Target variable: work interference + treatment
# Exclude rows where the original target-defining variables were missing
df = df_ml.copy()

# Keep only rows with observed work_interfere and treatment
if "work_interfere__missing" in df.columns:
    df = df[df["work_interfere__missing"] == 0].copy()

if "treatment__missing" in df.columns:
    df = df[df["treatment__missing"] == 0].copy()

# Create binary work interference variable
# 0 = Never
# 1 = Rarely / Sometimes / Often
df["work_interfere_binary"] = (
    df[[
        "work_interfere_Rarely",
        "work_interfere_Sometimes",
        "work_interfere_Often"
    ]].sum(axis=1) > 0
).astype(int)

# Create combined target variable:
# 0 = no interference
# 1 = interference but no treatment
# 2 = interference + treatment
df["combined_target"] = np.select(
    [
        df["work_interfere_Never"] == 1,
        (df["work_interfere_binary"] == 1) & (df["treatment"] == 0),
        (df["work_interfere_binary"] == 1) & (df["treatment"] == 1),
    ],
    [0, 1, 2],
    default=np.nan
)

# Drop any rows that still do not have a valid target
df = df.dropna(subset=["combined_target"]).copy()
df["combined_target"] = df["combined_target"].astype(int)

# Define predictors and target
# Remove:
# - target variable
# - target-defining variables (to avoid leakage)
# - missingness indicators for target-defining variables
X = df.drop(columns=[
    "combined_target",
    "treatment",
    "work_interfere_Never",
    "work_interfere_Rarely",
    "work_interfere_Sometimes",
    "work_interfere_Often",
    "work_interfere__missing",
    "work_interfere_binary",
    "treatment__missing"
], errors="ignore")

y = df["combined_target"]

# Check final class distribution
print(y.value_counts())
print(y.value_counts(normalize=True))
print(X.shape, y.shape)

combined_target
2    603
0    213
1    179
Name: count, dtype: int64
combined_target
2    0.606030
0    0.214070
1    0.179899
Name: proportion, dtype: float64
(995, 135) (995,)


In [19]:
# Define feature matrix X by removing:
# - the target variable (combined_target)
# - variables used to construct the target
# - intermediate variables related to work interference
# X = df.drop(columns=[
#     "combined_target",                # target variable
#     "treatment",                     # directly used in target construction
#     "work_interfere_Often",          # components of work_interfere
#     "work_interfere_Rarely",
#     "work_interfere_Sometimes",
#     "work_interfere__missing",       # missing indicator for target-related variable
#     "work_interfere_binary"          # intermediate variable used to build target
# ])
#
# # Define target variable
# y = df["combined_target"]
#
# # Check dimensions of feature matrix and target vector
# print(X.shape, y.shape)
#
# # Check for missing values in target
# # Should be 0 to ensure valid training
# print(y.isna().sum())
#
# # Inspect class distribution
# # Important for understanding class imbalance
# print(y.value_counts())

(1259, 135) (1259,)
0
combined_target
2    607
1    439
0    213
Name: count, dtype: int64


In [4]:
# Split the dataset into training and test sets
# - test_size=0.2 → 80% training, 20% testing
# - random_state ensures reproducibility
# - stratify=y preserves the class distribution in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Display shapes of training and test sets
# Confirms correct split and number of features
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Display normalized class distribution in the training set
# Verify that stratification preserved class proportions
print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))

Train shape: (796, 135)
Test shape: (199, 135)

Train class distribution:
combined_target
2    0.606784
0    0.213568
1    0.179648
Name: proportion, dtype: float64


In [5]:
# Initialize Random Forest model to be used for feature selection
# - n_estimators: number of trees in the forest
# - max_depth, min_samples_*: control model complexity and reduce overfitting
# - max_features="sqrt": standard choice for classification trees
# - class_weight="balanced": adjusts for class imbalance
# - n_jobs=-1: uses all available CPU cores
rf_selector = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

# Fit the Random Forest model on the training data
# This model will learn feature importances based on nonlinear relationships
rf_selector.fit(X_train, y_train)

# Initialize feature selector using the trained Random Forest
# - threshold="median": keep features with importance above the median importance
# This results in selecting approximately 50% of the most important features
selector = SelectFromModel(
    rf_selector,
    threshold="median"
)

# Fit the selector to identify important features
selector.fit(X_train, y_train)

# Transform training and test sets to keep only selected features
X_train_sel = selector.transform(X_train)
X_test_sel = selector.transform(X_test)

# Retrieve names of selected features
selected_features = X_train.columns[selector.get_support()]

# Display number and list of selected features
print("Number of selected features:", len(selected_features))
print("\nSelected features:")
print(selected_features.tolist())

Number of selected features: 68

Selected features:
['Age', 'self_employed', 'family_history', 'remote_work', 'tech_company', 'obs_consequence', 'Gender__missing', 'state__missing', 'self_employed__missing', 'Gender_male', 'Country_Canada', 'Country_Germany', 'Country_Ireland', 'Country_Netherlands', 'Country_Sweden', 'Country_United Kingdom', 'Country_United States', 'state_CA', 'state_FL', 'state_GA', 'state_IL', 'state_IN', 'state_MA', 'state_MI', 'state_MN', 'state_MO', 'state_NY', 'state_OH', 'state_OR', 'state_PA', 'state_TN', 'state_TX', 'state_VA', 'state_WA', 'state_WI', 'no_employees_100-500', 'no_employees_26-100', 'no_employees_500-1000', 'no_employees_6-25', 'no_employees_More than 1000', 'benefits_No', 'benefits_Yes', 'care_options_Not sure', 'care_options_Yes', 'wellness_program_No', 'wellness_program_Yes', 'seek_help_No', 'seek_help_Yes', 'anonymity_No', 'anonymity_Yes', 'leave_Somewhat difficult', 'leave_Somewhat easy', 'leave_Very difficult', 'leave_Very easy', 'menta

In [6]:
# Initialize Logistic Regression model for the hybrid approach
# - max_iter increased to ensure convergence
# - class_weight="balanced": compensates for class imbalance
# - this model serves as the interpretable component of the hybrid approach
logreg_hybrid = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

# Train the Logistic Regression model on the selected features
# These features were previously chosen by a nonlinear model (Random Forest)
logreg_hybrid.fit(X_train_sel, y_train)

# Generate predictions on the test set
y_pred_hybrid1 = logreg_hybrid.predict(X_test_sel)

# Evaluate model performance
# - Accuracy: overall correctness
# - Macro F1: treats all classes equally
# - Weighted F1: accounts for class frequency
print("Hybrid 1: RF Feature Selection -> Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_hybrid1))
print("Macro F1:", f1_score(y_test, y_pred_hybrid1, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_hybrid1, average="weighted"))

# Detailed classification report:
# shows precision, recall, and F1-score for each class separately
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_hybrid1))

# Confusion matrix:
# shows how predictions are distributed across true vs predicted classes
# useful for identifying which classes are confused with each other
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_hybrid1))

Hybrid 1: RF Feature Selection -> Logistic Regression
Accuracy: 0.5678391959798995
Macro F1: 0.5172574283320079
Weighted F1: 0.5887868544728371

Classification Report:

              precision    recall  f1-score   support

           0       0.41      0.60      0.49        43
           1       0.33      0.44      0.38        36
           2       0.83      0.59      0.69       120

    accuracy                           0.57       199
   macro avg       0.52      0.55      0.52       199
weighted avg       0.64      0.57      0.59       199


Confusion Matrix:

[[26 10  7]
 [12 16  8]
 [26 23 71]]


In [7]:
# Create a DataFrame of model coefficients from the multinomial Logistic Regression
# - rows correspond to classes (class_0, class_1, class_2)
# - columns correspond to the selected features
# - coefficients represent the effect of each feature on the log-odds of belonging to a class
coef_df = pd.DataFrame(
    logreg_hybrid.coef_,
    columns=selected_features,
    index=[f"class_{c}" for c in logreg_hybrid.classes_]
)

# Transpose the DataFrame so that features are rows and classes are columns
# Then sort features by the absolute value of their coefficient for class_2
# - class_2 corresponds to "interference + treatment"
# - sorting by absolute value highlights the most influential predictors (positive or negative)
# Finally, display the top 15 most important features for class_2
coef_df.T.sort_values(by="class_2", key=np.abs, ascending=False).head(15)

,class_0,class_1,class_2
family_history,-0.902363,-0.025975,0.928338
coworkers_Yes,-0.554632,-0.271212,0.825843
state_MO,0.515965,0.183067,-0.699031
state_OH,-0.533824,-0.087244,0.621068
no_employees_500-1000,0.711198,-0.109998,-0.601200
state_VA,0.241812,0.354144,-0.595956
state_MN,0.141160,-0.681103,0.539943
state__missing,-0.146859,0.676178,-0.529319
state_OR,-0.270289,-0.247756,0.518044
leave_Very difficult,-0.834534,0.334242,0.500292


In [8]:
# Initialize XGBoost classifier to be used for feature selection
# - n_estimators: number of trees
# - max_depth: controls tree complexity
# - learning_rate: step size for boosting
# - subsample, colsample_bytree: introduce randomness and reduce overfitting
# - objective="multi:softprob": multiclass classification with probability outputs
# - num_class=3: number of target classes
# - eval_metric="mlogloss": suitable loss for multiclass classification
# - tree_method="hist": efficient histogram-based algorithm
xgb_selector = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

# Fit XGBoost model on training data
# The model learns nonlinear relationships and assigns importance scores to features
xgb_selector.fit(X_train, y_train)

# Initialize feature selector using the trained XGBoost model
# - threshold="median": retains features with importance above the median
selector_xgb = SelectFromModel(
    xgb_selector,
    threshold="median"
)

# Fit selector to determine which features to keep
selector_xgb.fit(X_train, y_train)

# Transform training and test data to include only selected features
# This reduces dimensionality and keeps the most informative predictors
X_train_sel_xgb = selector_xgb.transform(X_train)
X_test_sel_xgb = selector_xgb.transform(X_test)

# Retrieve names of selected features
selected_features_xgb = X_train.columns[selector_xgb.get_support()]

# Display number and list of selected features
# Useful for comparison with Random Forest-based feature selection
print("Number of selected features:", len(selected_features_xgb))
print(selected_features_xgb.tolist())

Number of selected features: 68
['Age', 'self_employed', 'family_history', 'remote_work', 'tech_company', 'obs_consequence', 'Gender__missing', 'state__missing', 'self_employed__missing', 'Gender_male', 'Country_Canada', 'Country_Germany', 'Country_Ireland', 'Country_Italy', 'Country_Netherlands', 'Country_New Zealand', 'Country_Sweden', 'Country_United Kingdom', 'Country_United States', 'state_CA', 'state_GA', 'state_IL', 'state_IN', 'state_MA', 'state_MI', 'state_MN', 'state_MO', 'state_NY', 'state_OH', 'state_OR', 'state_PA', 'state_TN', 'state_TX', 'state_VA', 'state_WA', 'no_employees_100-500', 'no_employees_26-100', 'no_employees_500-1000', 'no_employees_6-25', 'no_employees_More than 1000', 'benefits_No', 'benefits_Yes', 'care_options_Not sure', 'care_options_Yes', 'wellness_program_No', 'wellness_program_Yes', 'seek_help_No', 'seek_help_Yes', 'anonymity_No', 'anonymity_Yes', 'leave_Somewhat difficult', 'leave_Somewhat easy', 'leave_Very difficult', 'leave_Very easy', 'mental_he

In [9]:
# Initialize Logistic Regression model for the second hybrid approach
# - Uses features selected by XGBoost (nonlinear model)
# - max_iter increased to ensure convergence
# - class_weight="balanced": handles class imbalance
logreg_hybrid_xgb = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

# Train Logistic Regression on XGBoost-selected features
# This combines nonlinear feature selection with a linear, interpretable model
logreg_hybrid_xgb.fit(X_train_sel_xgb, y_train)

# Generate predictions on the test set
y_pred_hybrid2 = logreg_hybrid_xgb.predict(X_test_sel_xgb)

# Evaluate model performance
# - Accuracy: overall correctness
# - Macro F1: treats all classes equally
# - Weighted F1: accounts for class distribution
print("Hybrid 2: XGB Feature Selection -> Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_hybrid2))
print("Macro F1:", f1_score(y_test, y_pred_hybrid2, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_hybrid2, average="weighted"))

# Detailed classification report:
# shows precision, recall, and F1-score for each class
# useful for identifying which classes are harder to predict
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_hybrid2))

# Confusion matrix:
# provides insight into misclassification patterns between classes
# helps analyze where the model makes systematic errors
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_hybrid2))

Hybrid 2: XGB Feature Selection -> Logistic Regression
Accuracy: 0.5678391959798995
Macro F1: 0.5146069584519616
Weighted F1: 0.5895186980601022

Classification Report:

              precision    recall  f1-score   support

           0       0.40      0.58      0.47        43
           1       0.33      0.44      0.38        36
           2       0.83      0.60      0.70       120

    accuracy                           0.57       199
   macro avg       0.52      0.54      0.51       199
weighted avg       0.64      0.57      0.59       199


Confusion Matrix:

[[25 11  7]
 [12 16  8]
 [26 22 72]]


In [10]:
# Define base models for stacking
# These represent different modeling paradigms:
# - Logistic Regression - linear, interpretable model
# - Random Forest - nonlinear, ensemble tree model
# - XGBoost - gradient boosting model with high predictive power
base_estimators = [
    ("lr", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )),
    ("rf", RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )),
    ("xgb", XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
]

# Initialize stacking classifier
# - base estimators generate predictions
# - final_estimator learns how to combine these predictions
# - stack_method="predict_proba": uses predicted probabilities as input to meta-model
# - cv=5: cross-validation used to generate out-of-fold predictions for training meta-model
stack_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=2000, random_state=42),
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1
)

# Train stacking model on training data
# The meta-model learns how to optimally combine base model outputs
stack_model.fit(X_train, y_train)

# Generate predictions on the test set
y_pred_stack = stack_model.predict(X_test)

# Evaluate performance of stacking hybrid model
# - Accuracy: overall correctness
# - Macro F1: equal importance to all classes
# - Weighted F1: accounts for class distribution
print("Hybrid 3: Stacking")
print("Accuracy:", accuracy_score(y_test, y_pred_stack))
print("Macro F1:", f1_score(y_test, y_pred_stack, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_stack, average="weighted"))

# Detailed classification report:
# shows per-class precision, recall, and F1-score
# useful for understanding performance differences across classes
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_stack))

# Confusion matrix:
# shows how predictions are distributed across true vs predicted classes
# helps identify systematic misclassification patterns
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_stack))

Hybrid 3: Stacking
Accuracy: 0.5979899497487438
Macro F1: 0.35468040731198625
Weighted F1: 0.5194430007971399

Classification Report:

              precision    recall  f1-score   support

           0       0.36      0.28      0.32        43
           1       0.00      0.00      0.00        36
           2       0.64      0.89      0.75       120

    accuracy                           0.60       199
   macro avg       0.34      0.39      0.35       199
weighted avg       0.47      0.60      0.52       199


Confusion Matrix:

[[ 12   0  31]
 [  8   0  28]
 [ 13   0 107]]


D:\Diana\Anul III\BachelorArbeit\WellBeingModelling\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Diana\Anul III\BachelorArbeit\WellBeingModelling\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Diana\Anul III\BachelorArbeit\WellBeingModelling\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

In [11]:
# Define base models for stacking, same approach as before
base_estimators = [
    ("lr", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",   # handles class imbalance at base level
        random_state=42
    )),
    ("rf", RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        class_weight="balanced",   # adjusts for imbalance
        random_state=42,
        n_jobs=-1
    )),
    ("xgb", XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
]

stack_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(
        max_iter=2000,
        class_weight="balanced"   # difference - add balanced for avoiding bias toward majority classes
    ),
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1
)

# Train stacking model
# The meta-model learns how to combine base model predictions
stack_model.fit(X_train, y_train)

# Generate predictions on the test set
y_pred_stack = stack_model.predict(X_test)

# Evaluate performance
# - Accuracy: overall correctness
# - Macro F1: equal importance to all classes (key metric for imbalanced data)
# - Weighted F1: weighted by class frequency
print("Hybrid 3: Stacking")
print("Accuracy:", accuracy_score(y_test, y_pred_stack))
print("Macro F1:", f1_score(y_test, y_pred_stack, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_stack, average="weighted"))

# Classification report:
# shows precision, recall, and F1-score for each class
# useful for analyzing class-specific performance
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_stack))

# Confusion matrix:
# shows how predictions are distributed across true vs predicted classes
# helps identify misclassification patterns and class overlap
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_stack))

Hybrid 3: Stacking
Accuracy: 0.5678391959798995
Macro F1: 0.48629425397586507
Weighted F1: 0.5925527903241935

Classification Report:

              precision    recall  f1-score   support

           0       0.37      0.47      0.41        43
           1       0.26      0.36      0.30        36
           2       0.84      0.67      0.74       120

    accuracy                           0.57       199
   macro avg       0.49      0.50      0.49       199
weighted avg       0.63      0.57      0.59       199


Confusion Matrix:

[[20 15  8]
 [16 13  7]
 [18 22 80]]


In [12]:
# Create a summary DataFrame to compare hybrid models
# Each row corresponds to one hybrid approach and its evaluation metrics
results = pd.DataFrame([
    {
        "Model": "RF selection -> Logistic Regression",
        # Overall accuracy of the hybrid model
        "Accuracy": accuracy_score(y_test, y_pred_hybrid1),
        # Macro F1: gives equal importance to all classes
        "Macro F1": f1_score(y_test, y_pred_hybrid1, average="macro"),
        # Weighted F1: accounts for class distribution
        "Weighted F1": f1_score(y_test, y_pred_hybrid1, average="weighted"),
    },
    {
        "Model": "XGB selection -> Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred_hybrid2),
        "Macro F1": f1_score(y_test, y_pred_hybrid2, average="macro"),
        "Weighted F1": f1_score(y_test, y_pred_hybrid2, average="weighted"),
    },
    {
        "Model": "Stacking (LR + RF + XGB)",
        "Accuracy": accuracy_score(y_test, y_pred_stack),
        "Macro F1": f1_score(y_test, y_pred_stack, average="macro"),
        "Weighted F1": f1_score(y_test, y_pred_stack, average="weighted"),
    }
])

# Sort models by Macro F1 score (descending)
# This highlights the best model in terms of balanced performance across classes
results.sort_values("Macro F1", ascending=False)

,Model,Accuracy,Macro F1,Weighted F1
0,RF selection -> Logistic Regression,0.567839,0.517257,0.588787
1,XGB selection -> Logistic Regression,0.567839,0.514607,0.589519
2,Stacking (LR + RF + XGB),0.567839,0.486294,0.592553
